In [35]:
from utils import get_dataset_lines

# Small Parsimony Problem

**Code Challenge**: Implement SmallParsimony to solve the Small Parsimony Problem.

**Input**: An integer $n$ followed by an adjacency list for a rooted binary tree with $n$ leaves labeled by DNA strings.

**Output**: The minimum parsimony score of this tree, followed by the adjacency list of a tree corresponding to labeling internal nodes by DNA strings in order to minimize the parsimony score of the tree. You may break ties however you like.

**Note**: Remember to run SmallParsimony on each individual index of the strings at the leaves of the tree.

**Sample Input**:

```
4
4->CAAATCCC
4->ATTGCGAC
5->CTGCGCTG
5->ATGGACGA
6->4
6->5
```

**Sample Output**:

```
16
ATTGCGAC->ATAGCCAC:2
ATAGACAA->ATAGCCAC:2
ATAGACAA->ATGGACTA:2
ATGGACGA->ATGGACTA:1
CTGCGCTG->ATGGACTA:4
ATGGACTA->CTGCGCTG:4
ATGGACTA->ATGGACGA:1
ATGGACTA->ATAGACAA:2
ATAGCCAC->CAAATCCC:5
ATAGCCAC->ATTGCGAC:2
ATAGCCAC->ATAGACAA:2
CAAATCCC->ATAGCCAC:5
```

In [36]:
def hamming_distance(s1, s2):
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def SmallParsimony(n, adjacency_list):
    tree = {}
    node_strings = {}
    all_nodes = set()
    leaf_counter = 0
    raw_edges = []
    
    for line in adjacency_list:
        line = line.strip()
        if not line: continue
        parts = line.split('->')
        parent = int(parts[0])
        child_str = parts[1]
        all_nodes.add(parent)
        
        if child_str.isdigit():
            child = int(child_str)
            all_nodes.add(child)
            raw_edges.append((parent, child))
        else:
            child = leaf_counter
            leaf_counter += 1
            node_strings[child] = child_str
            all_nodes.add(child)
            raw_edges.append((parent, child))
            
    for p, c in raw_edges:
        if p not in tree: tree[p] = []
        tree[p].append(c)
        
    children = set()
    for p in tree:
        for c in tree[p]:
            children.add(c)
            
    root = -1
    for node in all_nodes:
        if node not in children:
            root = node
            break
            
    alphabet = ['A', 'C', 'G', 'T']
    char_to_idx = {c: i for i, c in enumerate(alphabet)}
    k = len(next(iter(node_strings.values())))
    
    total_score = 0
    reconstructed_chars = {node: [] for node in all_nodes}
    
    post_order = []
    stack = [root]
    visited = set()
    while stack:
        node = stack[-1]
        if node not in tree:
            if node not in visited:
                visited.add(node)
                post_order.append(node)
            stack.pop()
        else:
            children_visited = True
            for child in tree[node]:
                if child not in visited:
                    stack.append(child)
                    children_visited = False
            if children_visited:
                if node not in visited:
                    visited.add(node)
                    post_order.append(node)
                stack.pop()
                
    for j in range(k):
        dp = {node: [float('inf')] * 4 for node in all_nodes}
        
        for node in all_nodes:
            if node in node_strings:
                char = node_strings[node][j]
                idx = char_to_idx[char]
                dp[node][idx] = 0
                
        for node in post_order:
            if node in tree:
                left = tree[node][0]
                right = tree[node][1]
                
                for current_char_idx in range(4):
                    min_left = float('inf')
                    for child_char_idx in range(4):
                        score = dp[left][child_char_idx]
                        if current_char_idx != child_char_idx:
                            score += 1
                        if score < min_left:
                            min_left = score
                            
                    min_right = float('inf')
                    for child_char_idx in range(4):
                        score = dp[right][child_char_idx]
                        if current_char_idx != child_char_idx:
                            score += 1
                        if score < min_right:
                            min_right = score
                            
                    dp[node][current_char_idx] = min_left + min_right
                    
        min_root_score = min(dp[root])
        total_score += min_root_score
        
        root_char_idx = -1
        for idx in range(4):
            if dp[root][idx] == min_root_score:
                root_char_idx = idx
                break
        
        reconstructed_chars[root].append(alphabet[root_char_idx])
        
        queue = [(root, root_char_idx)]
        while queue:
            parent, parent_char_idx = queue.pop(0)
            if parent in tree:
                for child in tree[parent]:
                    best_child_char_idx = -1
                    min_val = float('inf')
                    for c_idx in range(4):
                        cost = 1 if c_idx != parent_char_idx else 0
                        val = dp[child][c_idx] + cost
                        if val < min_val:
                            min_val = val
                            best_child_char_idx = c_idx
                    reconstructed_chars[child].append(alphabet[best_child_char_idx])
                    queue.append((child, best_child_char_idx))
                    
    final_strings = {}
    for node, chars in reconstructed_chars.items():
        final_strings[node] = "".join(chars)
        
    result_edges = []
    nodes_to_process = list(tree.keys())
    for u in nodes_to_process:
        for v in tree[u]:
            u_str = final_strings[u]
            v_str = final_strings[v]
            dist = hamming_distance(u_str, v_str)
            result_edges.append(f"{u_str}->{v_str}:{dist}")
            result_edges.append(f"{v_str}->{u_str}:{dist}")
            
    return total_score, result_edges

In [10]:
# Sample Input
n = 4
adjacency_list = [
    "4->CAAATCCC",
    "4->ATTGCGAC",
    "5->CTGCGCTG",
    "5->ATGGACGA",
    "6->4",
    "6->5"
]

score, edges = SmallParsimony(n, adjacency_list)
print(score)
for edge in edges:
    print(edge)

# Verification
# Note: There can be multiple valid parsimonious trees, so exact edge match might vary.
# But the score should be 16.
assert score == 16
print("Test passed!")

16
ATAGACAC->CAAATCCC:5
CAAATCCC->ATAGACAC:5
ATAGACAC->ATTGCGAC:3
ATTGCGAC->ATAGACAC:3
ATGGACAA->CTGCGCTG:5
CTGCGCTG->ATGGACAA:5
ATGGACAA->ATGGACGA:1
ATGGACGA->ATGGACAA:1
ATAGACAA->ATAGACAC:1
ATAGACAC->ATAGACAA:1
ATAGACAA->ATGGACAA:1
ATGGACAA->ATAGACAA:1
Test passed!


In [12]:
# Test Dataset
test_dataset_filename = 'dataset_30291_9.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    adjacency_list = lines[1:]
    
    score, edges = SmallParsimony(n, adjacency_list)
    print(score)
    for edge in edges:
        print(edge)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

13295
ACGCCGACGCGATCTACCCAAAGATCATAGCACACAAAAAGATCAAACATCCTACCGTCGGGAAATAAGATATTGTATCCAGGGAGGCAACGAAACGTTGCAGCGGTACGTCCAGCTCACTCCATTATAGAATGGGAAATGAATCGCAAACAGAACCAGAATACGCCCTCTTGCAAGAAACCAAGCGAA->GCGCCTGCGCGGGTTGCCCCACTTTCAAACCTCACAGTGGGTAGAAATTTTGTACCATAGGGAATGGTGATAGGATATGCTGGGAATGTACGATACTTCTAAGAGCGCCATCCTGTTCACTCTTTTATCCAGTGGGAAATTGCTCGCAAACTGAACCAGCTATCTCTCTCAAGCGAGAAATCGAGTGGT:75
GCGCCTGCGCGGGTTGCCCCACTTTCAAACCTCACAGTGGGTAGAAATTTTGTACCATAGGGAATGGTGATAGGATATGCTGGGAATGTACGATACTTCTAAGAGCGCCATCCTGTTCACTCTTTTATCCAGTGGGAAATTGCTCGCAAACTGAACCAGCTATCTCTCTCAAGCGAGAAATCGAGTGGT->ACGCCGACGCGATCTACCCAAAGATCATAGCACACAAAAAGATCAAACATCCTACCGTCGGGAAATAAGATATTGTATCCAGGGAGGCAACGAAACGTTGCAGCGGTACGTCCAGCTCACTCCATTATAGAATGGGAAATGAATCGCAAACAGAACCAGAATACGCCCTCTTGCAAGAAACCAAGCGAA:75
ACGCCGACGCGATCTACCCAAAGATCATAGCACACAAAAAGATCAAACATCCTACCGTCGGGAAATAAGATATTGTATCCAGGGAGGCAACGAAACGTTGCAGCGGTACGTCCAGCTCACTCCATTATAGAATGGGAAATGAATCGCAAACAGAACCAGAATACGCCCTCTTGCAAGAAACCAAGCGAA->AAGTCGATTCCCTCTCCCTAGAGATCCTGGCACAT

# Small Parsimony in an Unrooted Tree Problem

**Code Challenge**: Solve the Small Parsimony in an Unrooted Tree Problem.

**Input**: An integer $n$ followed by an adjacency list for an unrooted binary tree with $n$ leaves labeled by DNA strings.

**Output**: The minimum parsimony score of this tree, followed by the adjacency list of the tree corresponding to labeling internal nodes by DNA strings in order to minimize the parsimony score of the tree.

**Sample Input**:

```
4
TCGGCCAA->4
4->TCGGCCAA
CCTGGCTG->4
4->CCTGGCTG
CACAGGAT->5
5->CACAGGAT
TGAGTACC->5
5->TGAGTACC
4->5
5->4
```

**Sample Output**:

```
17
TCGGCCAA->CCAGGCAC:4
CCTGGCTG->CCAGGCAC:3
TGAGTACC->CAAGGAAC:4
CCAGGCAC->CCTGGCTG:3
CCAGGCAC->CAAGGAAC:2
CCAGGCAC->TCGGCCAA:4
CACAGGAT->CAAGGAAC:4
CAAGGAAC->CACAGGAT:4
CAAGGAAC->TGAGTACC:4
CAAGGAAC->CCAGGCAC:2
```

In [37]:
def SmallParsimonyUnrooted(n, lines):
    adj = {}
    nodes = set()
    
    for line in lines:
        line = line.strip()
        if not line: continue
        parts = line.split('->')
        u = parts[0]
        v = parts[1]
        if u not in adj: adj[u] = []
        adj[u].append(v)
        nodes.add(u)
        nodes.add(v)
        
    leaves = []
    internal_nodes = []
    for node in nodes:
        if len(adj.get(node, [])) == 1:
            leaves.append(node)
        else:
            internal_nodes.append(node)
            
    # Root the tree at an arbitrary internal edge to transform the unrooted problem into a rooted one.
    root_edge = None
    for u in internal_nodes:
        for v in adj[u]:
            if v in internal_nodes:
                root_edge = (u, v)
                break
        if root_edge: break
        
    if not root_edge:
        # Fallback: If no internal edge exists (e.g., star graph), root at an edge connected to a leaf.
        if internal_nodes:
            u = internal_nodes[0]
            v = adj[u][0]
            root_edge = (u, v)
        else:
            u = leaves[0]
            v = adj[u][0]
            root_edge = (u, v)
            
    u_root, v_root = root_edge
    
    tree = {}
    tree["ROOT"] = [u_root, v_root]
    
    # Construct the rooted tree structure using BFS starting from the chosen root edge.
    queue = [(u_root, "ROOT"), (v_root, "ROOT")]
    visited = {u_root, v_root}
    
    while queue:
        curr, p = queue.pop(0)
        
        children = []
        if curr in adj:
            for neighbor in adj[curr]:
                if neighbor == p: continue
                if (curr == u_root and neighbor == v_root) or (curr == v_root and neighbor == u_root):
                    continue
                children.append(neighbor)
                visited.add(neighbor)
                queue.append((neighbor, curr))
        
        if children:
            tree[curr] = children
            
    # Apply Sankoff's Algorithm to compute parsimony scores and ancestral sequences.
    k = len(leaves[0])
    alpha = "ACGT"
    char_to_idx = {c: i for i, c in enumerate(alpha)}
    
    node_labels = {node: [] for node in nodes}
    node_labels["ROOT"] = []
    
    total_score = 0
    
    def get_post_order(node, order):
        if node in tree:
            for child in tree[node]:
                get_post_order(child, order)
        order.append(node)
        
    post_order = []
    get_post_order("ROOT", post_order)
    
    for i in range(k):
        dp = {}
        
        for node in post_order:
            if node not in tree: # Base case: Leaf node in the rooted tree.
                if node in leaves:
                    char = node[i]
                    costs = [float('inf')] * 4
                    costs[char_to_idx[char]] = 0
                    dp[node] = costs
                else:
                    # Unreachable case: Internal nodes in the unrooted tree should not become leaves in the rooted tree unless the input is degenerate.
                    pass
            else:
                left = tree[node][0]
                right = tree[node][1]
                
                costs = []
                for curr_c_idx in range(4):
                    c_left = float('inf')
                    for child_c_idx in range(4):
                        cost = dp[left][child_c_idx] + (0 if curr_c_idx == child_c_idx else 1)
                        if cost < c_left: c_left = cost
                    
                    c_right = float('inf')
                    for child_c_idx in range(4):
                        cost = dp[right][child_c_idx] + (0 if curr_c_idx == child_c_idx else 1)
                        if cost < c_right: c_right = cost
                        
                    costs.append(c_left + c_right)
                dp[node] = costs
                
        root_costs = dp["ROOT"]
        min_root_cost = min(root_costs)
        total_score += min_root_cost
        
        root_char_idx = -1
        for idx in range(4):
            if root_costs[idx] == min_root_cost:
                root_char_idx = idx
                break
        
        node_labels["ROOT"].append(alpha[root_char_idx])
        
        queue_bt = [("ROOT", root_char_idx)]
        while queue_bt:
            parent, p_char_idx = queue_bt.pop(0)
            
            if parent in tree:
                for child in tree[parent]:
                    best_c_idx = -1
                    min_c_cost = float('inf')
                    
                    for c_idx in range(4):
                        cost = dp[child][c_idx] + (0 if p_char_idx == c_idx else 1)
                        if cost < min_c_cost:
                            min_c_cost = cost
                            best_c_idx = c_idx
                    
                    node_labels[child].append(alpha[best_c_idx])
                    queue_bt.append((child, best_c_idx))
            else:
                if not node_labels[parent]:
                     node_labels[parent].append(parent[i])

    final_labels = {node: "".join(chars) for node, chars in node_labels.items()}
    
    for leaf in leaves:
        final_labels[leaf] = leaf
        
    result_lines = []
    result_lines.append(str(total_score))
    
    def hamming(s1, s2):
        return sum(1 for a, b in zip(s1, s2) if a != b)
        
    for u in adj:
        u_label = final_labels[u]
        for v in adj[u]:
            v_label = final_labels[v]
            dist = hamming(u_label, v_label)
            result_lines.append(f"{u_label}->{v_label}:{dist}")
            
    return result_lines

In [14]:
# Sample Input
n = 4
lines = [
    "TCGGCCAA->4",
    "4->TCGGCCAA",
    "CCTGGCTG->4",
    "4->CCTGGCTG",
    "CACAGGAT->5",
    "5->CACAGGAT",
    "TGAGTACC->5",
    "5->TGAGTACC",
    "4->5",
    "5->4"
]

# Run the function
result = SmallParsimonyUnrooted(n, lines)

# Display the results
for line in result:
    print(line)

# Test Assertion
expected_score = 17
calculated_score = int(result[0])
assert calculated_score == expected_score, f"Expected score {expected_score}, but got {calculated_score}"

# Note: Internal node labels may vary due to multiple optimal solutions. Verification focuses on the parsimony score.
print("Test passed!")

17
TCGGCCAA->CCAGGCAA:3
CCAGGCAA->TCGGCCAA:3
CCAGGCAA->CCTGGCTG:3
CCAGGCAA->CAAGGAAA:2
CCTGGCTG->CCAGGCAA:3
CACAGGAT->CAAGGAAA:4
CAAGGAAA->CACAGGAT:4
CAAGGAAA->TGAGTACC:5
CAAGGAAA->CCAGGCAA:2
TGAGTACC->CAAGGAAA:5
Test passed!


In [15]:
# Test Dataset
test_dataset_filename = 'dataset_30291_11.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    adjacency_list = lines[1:]
    
    result = SmallParsimonyUnrooted(n, adjacency_list)
    for line in result:
        print(line)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

506
GATACCCTCAGCACCCGGTGTACTACCACT->GATAGCCTCATTAACCGCAATAATACCACT:8
GATAGCCTCATTAACCGCAATAATACCACT->GATACCCTCAGCACCCGGTGTACTACCACT:8
GATAGCCTCATTAACCGCAATAATACCACT->TTTAGCCTCCTTTACCGCACAGATCCTACT:9
GATAGCCTCATTAACCGCAATAATACCACT->GATAGCTTTATTAACCCCAATAACACCAAC:6
TTTAGCCTCCTTTACCGCACAGATCCTACT->GATAGCCTCATTAACCGCAATAATACCACT:9
GAAAACACACGCGGACGCAATACATTCAGG->GATAATATACTCGCACCTAATAAATACAAG:10
GATAATATACTCGCACCTAATAAATACAAG->GAAAACACACGCGGACGCAATACATTCAGG:10
GATAATATACTCGCACCTAATAAATACAAG->GATCCTGTACTCGTTCTTAACCAGCAGCCG:13
GATAATATACTCGCACCTAATAAATACAAG->AATAATATAATTGCACCTAATAACTCCAAG:5
GATCCTGTACTCGTTCTTAACCAGCAGCCG->GATAATATACTCGCACCTAATAAATACAAG:13
CTTAGTTGAAGGGCTGCTACAGTCAGTCTT->ATTAATTGAATTGCAGCTAAAAACTCTATA:12
ATTAATTGAATTGCAGCTAAAAACTCTATA->CTTAGTTGAAGGGCTGCTACAGTCAGTCTT:12
ATTAATTGAATTGCAGCTAAAAACTCTATA->ATAAAGTGTGTTCGCTATGTAAAGTCTATA:12
ATTAATTGAATTGCAGCTAAAAACTCTATA->ATTAATAGAATTGCAGCTAATAACTCTAAG:4
ATAAAGTGTGTTCGCTATGTAAAGTCTATA->ATTAATTGAATTGCAGCTAAAAACTCTATA:12
ATGAAATGTCTTA

# Nearest Neighbors of a Tree Problem

**Code Challenge**: Solve the Nearest Neighbors of a Tree Problem.

**Input**: Two internal nodes $a$ and $b$ specifying an edge $e$, followed by an adjacency list of an unrooted binary tree.

**Output**: Two adjacency lists representing the nearest neighbors of the tree with respect to $e$. Separate the adjacency lists with a blank line.

**Sample Input**:

```
5 4
0->4
4->0
1->4
4->1
2->5
5->2
3->5
5->3
4->5
5->4
```

**Sample Output**:

```
1->4
0->5
3->4
2->5
5->2
5->4
5->0
4->1
4->5
4->3

1->5
0->4
3->4
2->5
5->2
5->4
5->1
4->0
4->5
4->3
```

In [38]:
def NearestNeighborsOfTree(a, b, adj):
    neighbors_a = [n for n in adj[a] if n != b]
    neighbors_b = [n for n in adj[b] if n != a]
    
    if len(neighbors_a) != 2 or len(neighbors_b) != 2:
        # In a binary tree, internal nodes have degree 3.
        # Excluding the edge (a,b), they should have 2 neighbors.
        return []

    results = []
    
    # We generate two neighbors by swapping one neighbor of a with each of neighbors_b.
    # Based on the sample output, we keep neighbors_a[0] fixed and swap neighbors_a[1].
    
    neighbor_to_swap = neighbors_a[1]
    
    swap_moves = [
        (neighbor_to_swap, neighbors_b[0]),
        (neighbor_to_swap, neighbors_b[1])
    ]
    
    for u, v in swap_moves:
        # Deep copy
        new_adj = {node: list(neighbors) for node, neighbors in adj.items()}
        
        # Swap u (neighbor of a) and v (neighbor of b)
        # Remove a-u, u-a
        new_adj[a].remove(u)
        new_adj[u].remove(a)
        
        # Remove b-v, v-b
        new_adj[b].remove(v)
        new_adj[v].remove(b)
        
        # Add a-v, v-a
        new_adj[a].append(v)
        new_adj[v].append(a)
        
        # Add b-u, u-b
        new_adj[b].append(u)
        new_adj[u].append(b)
        
        results.append(new_adj)
        
    return results

In [28]:
# Sample Input
a, b = 5, 4
adjacency_list = [
    "0->4", "4->0",
    "1->4", "4->1",
    "2->5", "5->2",
    "3->5", "5->3",
    "4->5", "5->4"
]

adj = {}
for line in adjacency_list:
    u, v = map(int, line.split('->'))
    if u not in adj: adj[u] = []
    adj[u].append(v)

results = NearestNeighborsOfTree(a, b, adj)

# Expected Output String
expected_output_str = """1->4
0->5
3->4
2->5
5->2
5->4
5->0
4->1
4->5
4->3

1->5
0->4
3->4
2->5
5->2
5->4
5->1
4->0
4->5
4->3"""

def print_adj(adj):
    for u in sorted(adj.keys()):
        for v in sorted(adj[u]): # Sorting v for consistent output
             print(f"{u}->{v}")

def parse_adj_list(adj_str):
    adj = {}
    for line in adj_str.strip().split('\n'):
        if not line.strip(): continue
        u, v = map(int, line.split('->'))
        if u not in adj: adj[u] = []
        adj[u].append(v)
    return adj

def normalize_adj(adj):
    # Sort keys and values for consistent representation
    normalized = []
    for u in sorted(adj.keys()):
        neighbors = sorted(adj[u])
        for v in neighbors:
            normalized.append(f"{u}->{v}")
    return "\n".join(normalized)

# Parse expected output
expected_blocks = expected_output_str.strip().split('\n\n')
expected_adjs = [parse_adj_list(block) for block in expected_blocks]

# Normalize and sort both lists of results to ensure order independence
actual_normalized = sorted([normalize_adj(res) for res in results])
expected_normalized = sorted([normalize_adj(res) for res in expected_adjs])

# Print results
print("Actual Output (Sorted):")
for res_str in actual_normalized:
    print(res_str)
    print()

print("Expected Output (Sorted):")
for res_str in expected_normalized:
    print(res_str)
    print()

# Test Assertion
assert actual_normalized == expected_normalized, "Actual output does not match expected output!"
print("Test passed!")

Actual Output (Sorted):
0->4
1->5
2->5
3->4
4->0
4->3
4->5
5->1
5->2
5->4

0->5
1->4
2->5
3->4
4->1
4->3
4->5
5->0
5->2
5->4

Expected Output (Sorted):
0->4
1->5
2->5
3->4
4->0
4->3
4->5
5->1
5->2
5->4

0->5
1->4
2->5
3->4
4->1
4->3
4->5
5->0
5->2
5->4

Test passed!


In [29]:
# Test Dataset
test_dataset_filename = 'dataset_30292_6.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    first_line = lines[0].split()
    a = int(first_line[0])
    b = int(first_line[1])
    
    adj = {}
    for line in lines[1:]:
        line = line.strip()
        if not line: continue
        u, v = map(int, line.split('->'))
        if u not in adj: adj[u] = []
        adj[u].append(v)
        
    results = NearestNeighborsOfTree(a, b, adj)
    
    for i, res in enumerate(results):
        print_adj(res)
        if i < len(results) - 1:
            print()
            
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0->32
1->32
2->33
3->33
4->34
5->34
6->35
7->35
8->36
9->36
10->37
11->37
12->38
13->38
14->39
15->39
16->40
17->51
18->41
19->41
20->42
21->42
22->43
23->43
24->44
25->44
26->45
27->45
28->46
29->46
30->47
31->47
32->0
32->1
32->48
33->2
33->3
33->53
34->4
34->5
34->55
35->6
35->7
35->60
36->8
36->9
36->50
37->10
37->11
37->48
38->12
38->13
38->57
39->14
39->15
39->40
40->16
40->39
40->51
41->18
41->19
41->55
42->20
42->21
42->50
43->22
43->23
43->52
44->24
44->25
44->49
45->26
45->27
45->49
46->28
46->29
46->58
47->30
47->31
47->54
48->32
48->37
48->57
49->44
49->45
49->52
50->36
50->42
50->58
51->17
51->40
51->53
52->43
52->49
52->54
53->33
53->51
53->56
54->47
54->52
54->60
55->34
55->41
55->56
56->53
56->55
56->59
57->38
57->48
57->59
58->46
58->50
58->61
59->56
59->57
59->61
60->35
60->54
60->61
61->58
61->59
61->60

0->32
1->32
2->33
3->33
4->34
5->34
6->35
7->35
8->36
9->36
10->37
11->37
12->38
13->38
14->39
15->39
16->40
17->51
18->41
19->41
20->42
21->42
22->43
23->43
24->44


# Large Parsimony Problem
**Code Challenge**: Implement the nearest neighbor interchange heuristic for the Large Parsimony Problem.

**Input**: An integer $n$, followed by an adjacency list for an unrooted binary tree whose $n$ leaves are labeled by DNA strings and whose internal nodes are labeled by integers.

**Output**: The parsimony score and unrooted labeled tree obtained after every step of the nearest neighbor interchange heuristic. Each step should be separated by a blank line.

**Sample Input**:

```
5
GCAGGGTA->5
TTTACGCG->5
CGACCTGA->6
GATTCCAC->6
5->TTTACGCG
5->GCAGGGTA
5->7
TCCGTAGT->7
7->5
7->6
7->TCCGTAGT
6->GATTCCAC
6->CGACCTGA
6->7
```

**Sample Output**:

```
22
TCCGTAGT->TCAGCGGA:4
GATTCCAC->GAACCCGA:4
CGACCTGA->GAACCCGA:3
TTTACGCG->TCAGCGGA:5
TCAGCGGA->TTTACGCG:5
TCAGCGGA->GCAGCGGA:1
TCAGCGGA->TCCGTAGT:4
GCAGGGTA->GCAGCGGA:2
GCAGCGGA->GAACCCGA:3
GCAGCGGA->GCAGGGTA:2
GCAGCGGA->TCAGCGGA:1
GAACCCGA->GATTCCAC:4
GAACCCGA->CGACCTGA:3
GAACCCGA->GCAGCGGA:3

21
TCCGTAGT->TCTGCGGA:4
GATTCCAC->GCTGCGGA:5
CGACCTGA->GCAGCGGA:4
TTTACGCG->TCTGCGGA:4
TCTGCGGA->TTTACGCG:4
TCTGCGGA->GCTGCGGA:1
TCTGCGGA->TCCGTAGT:4
GCAGGGTA->GCAGCGGA:2
GCTGCGGA->GCAGCGGA:1
GCTGCGGA->GATTCCAC:5
GCTGCGGA->TCTGCGGA:1
GCAGCGGA->CGACCTGA:4
GCAGCGGA->GCAGGGTA:2
GCAGCGGA->GCTGCGGA:1
```

In [43]:
def LargeParsimony(n, adj_lines):
    history = []
    
    # 1. Parse initial input to an adjacency list topology (preserving Node IDs)
    current_adj = {}
    for line in adj_lines:
        line = line.strip()
        if not line: continue
        parts = line.split('->')
        u, v = parts[0], parts[1]
        
        if u not in current_adj: current_adj[u] = []
        current_adj[u].append(v)
        
    # 2. Get initial score and labeled text output
    # SmallParsimonyUnrooted handles the tree rooting and Sankoff algorithm
    current_res = SmallParsimonyUnrooted(n, adj_lines)
    current_score = int(current_res[0])
    
    # 3. Iterative Improvement
    while True:
        best_score = current_score
        best_adj = None
        best_res_lines = None
        
        # Identify internal edges (u, v)
        # In an unrooted binary tree with leaves as labels:
        # Leaves have degree 1 usually (but here stored as adjacency list 'Leaf'->'Internal')
        # Internal nodes have degree 3.
        # We need edges between two internal nodes.
        
        edges_to_check = []
        seen_edges = set()
        
        for u in current_adj:
            for v in current_adj[u]:
                # canonical edge
                u_str, v_str = str(u), str(v)
                if u_str > v_str: edge = (v_str, u_str)
                else: edge = (u_str, v_str)
                
                if edge in seen_edges: continue
                seen_edges.add(edge)
                
                # Check if internal edge: both endpoints must have degree > 1
                if len(current_adj[u]) > 1 and len(current_adj[v]) > 1:
                    edges_to_check.append((u, v))
                    
        # Explore neighbors
        for u, v in edges_to_check:
            # Generate neighbors for this edge
            neighbors = NearestNeighborsOfTree(u, v, current_adj)
            
            for nb_adj in neighbors:
                # Convert nb_adj back to lines format for SmallParsimonyUnrooted
                nb_lines = []
                for node in nb_adj:
                    for neighbor in nb_adj[node]:
                        nb_lines.append(f"{node}->{neighbor}")
                        
                # Evaluate
                res = SmallParsimonyUnrooted(n, nb_lines)
                score = int(res[0])
                
                if score < best_score:
                    best_score = score
                    best_adj = nb_adj
                    best_res_lines = res
                    
        # Update or Break
        if best_score < current_score:
            current_score = best_score
            current_adj = best_adj  # Move to the better topology
            history.append(best_res_lines)
        else:
            break
            
    return history

def print_parsimony_history(history):
    for i, result_lines in enumerate(history):
        if i > 0:
            print()
        for line in result_lines:
            print(line)

In [44]:
# Sample Input
n = 5
adj_lines = [
    "GCAGGGTA->5",
    "TTTACGCG->5",
    "CGACCTGA->6",
    "GATTCCAC->6",
    "5->TTTACGCG",
    "5->GCAGGGTA",
    "5->7",
    "TCCGTAGT->7",
    "7->5",
    "7->6",
    "7->TCCGTAGT",
    "6->GATTCCAC",
    "6->CGACCTGA",
    "6->7"
]

history = LargeParsimony(n, adj_lines)
print_parsimony_history(history)

22
GCAGGGTA->GCAGCGGA:2
TTTACGCG->TCAGCGGA:5
CGACCTGA->GAACCCGA:3
GATTCCAC->GAACCCGA:4
TCAGCGGA->TTTACGCG:5
TCAGCGGA->GCAGCGGA:1
TCAGCGGA->TCCGTAGT:4
TCCGTAGT->TCAGCGGA:4
GCAGCGGA->TCAGCGGA:1
GCAGCGGA->GAACCCGA:3
GCAGCGGA->GCAGGGTA:2
GAACCCGA->GATTCCAC:4
GAACCCGA->CGACCTGA:3
GAACCCGA->GCAGCGGA:3

21
GCAGGGTA->GCAGCGGA:2
TTTACGCG->TCTGCGGA:4
CGACCTGA->GCAGCGGA:4
GATTCCAC->GCTGCGGA:5
TCTGCGGA->TTTACGCG:4
TCTGCGGA->GCTGCGGA:1
TCTGCGGA->TCCGTAGT:4
TCCGTAGT->TCTGCGGA:4
GCTGCGGA->TCTGCGGA:1
GCTGCGGA->GCAGCGGA:1
GCTGCGGA->GATTCCAC:5
GCAGCGGA->CGACCTGA:4
GCAGCGGA->GCTGCGGA:1
GCAGCGGA->GCAGGGTA:2


In [45]:
# Test Dataset
test_dataset_filename = 'dataset_30292_8.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    adj_lines = lines[1:]
    history = LargeParsimony(n, adj_lines)
    print_parsimony_history(history)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

147
ATCTCGTCCTCAGCCAATCTTACGCACAGGGAGATCATAT->ACCTTAGCCTCAGACCATTTTACGAACAACGCTGTAATTA:16
ACCTTAGCCTCAGACCATTTTACGAACAACGCTGTAATTA->ATCTCGTCCTCAGCCAATCTTACGCACAGGGAGATCATAT:16
ACCTTAGCCTCAGACCATTTTACGAACAACGCTGTAATTA->ACGTTAGCATCTTAACCTTCTCCGAATAACTCTGTAGGTA:12
ACCTTAGCCTCAGACCATTTTACGAACAACGCTGTAATTA->AGCGTAGGCGCAGACCAATTTTCGAACAACGCTGGTATTA:8
ACGTTAGCATCTTAACCTTCTCCGAATAACTCTGTAGGTA->ACCTTAGCCTCAGACCATTTTACGAACAACGCTGTAATTA:12
AGGTCAGTGGAAACGACGTATGAGCAAAGTCCGTGTGTGT->AGGGTAGTGGAAAGCCAATATTAAAACAGTTCTCGTATTG:17
AGGGTAGTGGAAAGCCAATATTAAAACAGTTCTCGTATTG->AGGTCAGTGGAAACGACGTATGAGCAAAGTCCGTGTGTGT:17
AGGGTAGTGGAAAGCCAATATTAAAACAGTTCTCGTATTG->AGCGTAGGCGCAAGCCAATTTTCAAACAGCTCTCGTATTG:7
AGGGTAGTGGAAAGCCAATATTAAAACAGTTCTCGTATTG->AGGGTAGTGGTAAGCCAACATTAAAACCGTTCTCATATTG:4
GGCGTCGGCGCTCGCCACTTCTCAGCGGGCTTTCGTCAAG->AGCGTAGGCGCAAGCCAATTTTCAAACAGCTCTCGTATTG:14
ACGGTCGAGTTGTGTTAACATTAAAACCGTACTGATATTC->AGGGTAGTGGTAAGCCAACATTAAAACCGTTCTCATATTG:11
AGGGTAGTGGTAAGCCAACATTAAAACCGTTCTCATATTG->ACGGTCGAGTT